# Project 9: Yelp Reviews NLP & ML Analysis

**Goal**: Generate and analyze a large synthetic Yelp-like review dataset. Build TF-IDF text vectorization, dimensionality reduction with TruncatedSVD, sentiment classification pipelines, topic discovery, and business recommendation scoring.

**Data**: ~500 MB synthetic review dataset (2M reviews across 50K businesses)

**Cash Stress Points**: Large text vectorization (TF-IDF sparse matrices), sparse matrix caching, ML pipeline object serialization, iterative model tuning, feature engineering chains

In [ ]:
%load_ext cash
%cash_on
%cash_badge print
%cash_debug on

In [ ]:
import numpy as np
import pandas as pd
import os
import time

# ML / NLP imports

print(f"NumPy {np.__version__}, pandas {pd.__version__}")
print(f"Working directory: {os.getcwd()}")


In [ ]:
%%cash

# ── Generate synthetic Yelp-like review dataset ──
# Target: ~2M reviews across 50K businesses (~400-500 MB in memory)

_rng = np.random.default_rng(42)
_n_reviews = 2_000_000
_n_businesses = 50_000

# Business attributes
_categories = [
    'Restaurants', 'Shopping', 'Food', 'Beauty & Spas', 'Health & Medical',
    'Home Services', 'Nightlife', 'Bars', 'Automotive', 'Event Planning',
    'Active Life', 'Hotels & Travel', 'Arts & Entertainment', 'Pets',
    'Education', 'Professional Services', 'Financial Services', 'Real Estate',
    'Local Services', 'Public Services'
]
_cities = [
    'Las Vegas', 'Phoenix', 'Toronto', 'Charlotte', 'Scottsdale',
    'Pittsburgh', 'Montreal', 'Mesa', 'Henderson', 'Tempe',
    'Chandler', 'Cleveland', 'Glendale', 'Madison', 'Gilbert'
]

# Review text templates by sentiment (positive / neutral / negative)
_positive_phrases = [
    "Absolutely amazing experience", "Best place in town", "Highly recommend",
    "Outstanding service and quality", "Will definitely come back",
    "Five stars all the way", "Exceeded all expectations", "A hidden gem",
    "Friendly staff and great atmosphere", "Perfect for date night",
    "The food was incredible", "Great value for money", "Love this place",
    "Top notch quality", "Never disappoints", "Wonderful ambiance",
    "Fast and efficient service", "Clean and well maintained",
]
_neutral_phrases = [
    "It was okay nothing special", "Average experience overall",
    "Not bad but could be better", "Decent place for the price",
    "Has room for improvement", "Standard quality nothing more",
    "Met expectations but didn't exceed", "Fair enough for what it is",
    "Some things were good others not so much", "Middle of the road",
]
_negative_phrases = [
    "Terrible experience never again", "Worst service I've ever had",
    "Complete waste of money", "Rude staff and dirty place",
    "Would give zero stars if I could", "Food was cold and tasteless",
    "Waited over an hour for nothing", "Disgusting conditions",
    "Management doesn't care at all", "Overpriced and underwhelming",
    "Not worth the drive", "Disappointed with the quality",
]

t0 = time.time()

# Generate business IDs
business_ids = np.array([f"biz_{i:06d}" for i in range(_n_businesses)])
business_categories = _rng.choice(_categories, size=_n_businesses)
business_cities = _rng.choice(_cities, size=_n_businesses)
business_avg_stars = _rng.normal(3.5, 0.8, size=_n_businesses).clip(1, 5)

# Generate reviews
review_business_idx = _rng.integers(0, _n_businesses, size=_n_reviews)
review_stars = np.zeros(_n_reviews, dtype=np.int8)
review_texts = []

# Star ratings influenced by business quality
_business_means = business_avg_stars[review_business_idx]
_noise = _rng.normal(0, 1, size=_n_reviews)
_raw_stars = (_business_means + _noise).clip(1, 5)
review_stars = np.round(_raw_stars).astype(np.int8)

# Generate review texts based on star ratings
for _i in range(_n_reviews):
    _s = review_stars[_i]
    if _s >= 4:
        _phrases = _rng.choice(_positive_phrases, size=_rng.integers(2, 5))
    elif _s == 3:
        _phrases = _rng.choice(_neutral_phrases, size=_rng.integers(2, 4))
    else:
        _phrases = _rng.choice(_negative_phrases, size=_rng.integers(2, 5))
    review_texts.append(". ".join(_phrases) + ".")

# User IDs
user_ids = np.array([f"user_{_rng.integers(0, 500_000):06d}" for _ in range(_n_reviews)])

# Build main DataFrame
reviews_df = pd.DataFrame({
    'review_id': [f"rev_{i:07d}" for i in range(_n_reviews)],
    'business_id': business_ids[review_business_idx],
    'user_id': user_ids,
    'stars': review_stars,
    'text': review_texts,
    'category': business_categories[review_business_idx],
    'city': business_cities[review_business_idx],
    'useful_votes': _rng.integers(0, 50, size=_n_reviews),
    'funny_votes': _rng.integers(0, 20, size=_n_reviews),
    'cool_votes': _rng.integers(0, 30, size=_n_reviews),
    'date': pd.date_range('2018-01-01', periods=_n_reviews, freq='15s'),
})

elapsed = time.time() - t0
_mem_mb = reviews_df.memory_usage(deep=True).sum() / 1e6
print(f"Generated {len(reviews_df):,} reviews across {_n_businesses:,} businesses")
print(f"DataFrame memory: {_mem_mb:.1f} MB")
print(f"Star distribution:\n{reviews_df['stars'].value_counts().sort_index().to_string()}")
print(f"Generation time: {elapsed:.1f}s")


In [ ]:
%%cash
# Quick diagnostic
print(f"reviews_df shape: {reviews_df.shape}")
print(f"Memory: {reviews_df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(reviews_df.head(3).to_string())
